# SWE-Finetune: Multi-Phase Training for SWE-bench & TerminalBench

This notebook trains Qwen3-30B-A3B using Tinker API for optimal performance on:
- **SWE-bench**: Software engineering agent tasks
- **TerminalBench**: Terminal/shell command tasks

## Phases
1. **Coding Foundation** - Magicoder, Evol-Instruct (~155K)
2. **Terminal/Shell** - NL-SHELL-MULTI, NL2SH-ALFA (~145K)
3. **Tool-Use** - xLAM, Glaive function calling (~180K)
4. **SWE-bench Trajectories** - Agent traces (~156K) **CRITICAL**
5. **Competitive Programming** - TACO, CodeForces (~35K)

## 1. Setup

In [ ]:
# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q tinker tinker-cookbook datasets transformers

In [ ]:
# Clone the repo (or upload files)
!git clone https://github.com/YOUR_USERNAME/swe-finetune.git /content/swe-finetune 2>/dev/null || echo 'Repo exists'
import sys
sys.path.insert(0, '/content/swe-finetune')

In [ ]:
# Set Tinker API key
import os
os.environ['TINKER_API_KEY'] = 'YOUR_API_KEY_HERE'  # Replace with your key

## 2. Configuration

In [ ]:
#@title Select Training Phase
PHASE = 4  #@param [1, 2, 3, 4, 5] {type:"integer"}
RESUME_FROM_CHECKPOINT = True  #@param {type:"boolean"}
MAX_SAMPLES_PER_DATASET = None  #@param {type:"raw"}

# Phase descriptions
PHASE_INFO = {
    1: "Coding Foundation (Magicoder, Evol-Instruct)",
    2: "Terminal/Shell (NL-SHELL-MULTI, NL2SH-ALFA)",
    3: "Tool-Use (xLAM, Glaive function calling)",
    4: "SWE-bench Trajectories (CRITICAL!)",
    5: "Competitive Programming (TACO, CodeForces)",
}
print(f"Selected: Phase {PHASE} - {PHASE_INFO[PHASE]}")

In [ ]:
# Load phase config
from configs import (
    phase1_config, phase2_config, phase3_config, 
    phase4_config, phase5_config
)

CONFIGS = {
    1: phase1_config,
    2: phase2_config,
    3: phase3_config,
    4: phase4_config,
    5: phase5_config,
}

config = CONFIGS[PHASE]
print(f"Config: {config.name}")
print(f"  LR: {config.training.learning_rate}")
print(f"  Batch: {config.training.batch_size}")
print(f"  Max Length: {config.training.max_length}")
print(f"  Epochs: {config.training.num_epochs}")

## 3. Initialize Tinker

In [ ]:
import tinker
from scripts.training import create_training_client
from scripts.preprocessing import get_tokenizer_and_renderer

# Create training client
training_client = create_training_client(
    model_name=config.model.name,
    lora_rank=config.model.lora_rank,
)
print(f"Created training client for {config.model.name}")

# Get tokenizer and renderer
tokenizer, renderer = get_tokenizer_and_renderer(
    model_name=config.model.name,
    max_length=config.training.max_length,
)
print("Tokenizer and renderer ready")

In [ ]:
# Resume from checkpoint if requested
from scripts.training.utils import find_latest_checkpoint, load_from_drive

checkpoint_dir = f"/content/drive/MyDrive/swe-finetune/checkpoints/{config.name}"
start_step = 0

if RESUME_FROM_CHECKPOINT:
    checkpoint = find_latest_checkpoint(checkpoint_dir)
    if checkpoint:
        print(f"Resuming from: {checkpoint}")
        training_client.load_state_with_optimizer(checkpoint)
        # Extract step number
        try:
            start_step = int(checkpoint.split('step_')[1].split('/')[0])
        except:
            start_step = 0
        print(f"Starting from step {start_step}")
    else:
        print("No checkpoint found, starting fresh")

## 4. Load Data

In [ ]:
# Load data for selected phase
from scripts.data_loaders import (
    load_coding_datasets,
    load_terminal_datasets,
    load_tooluse_datasets,
    load_swebench_trajectories,
    load_competitive_datasets,
)

LOADERS = {
    1: load_coding_datasets,
    2: load_terminal_datasets,
    3: load_tooluse_datasets,
    4: load_swebench_trajectories,
    5: load_competitive_datasets,
}

print(f"Loading data for Phase {PHASE}...")
data_iterator = LOADERS[PHASE](
    streaming=True,
    max_samples_per_dataset=MAX_SAMPLES_PER_DATASET,
    shuffle=True,
)

# Convert to list for counting (optional, uses memory)
# data = list(data_iterator)
# print(f"Loaded {len(data)} examples")

## 5. Training Loop

In [ ]:
from scripts.training import train_phase
from scripts.training.utils import save_to_drive
import os

# Checkpoint callback to save to Drive
def on_checkpoint(step, path):
    drive_path = f"swe-finetune/checkpoints/{config.name}/step_{step:06d}"
    save_to_drive(path, drive_path)

# Training config
train_config = {
    "learning_rate": config.training.learning_rate,
    "batch_size": config.training.batch_size,
    "checkpoint_every": config.training.checkpoint_every,
    "lr_schedule": config.training.lr_schedule,
    "warmup_steps": config.training.warmup_steps,
    "train_on_what": "last" if PHASE == 4 else "all",  # Last for trajectories
}

print(f"Starting training for Phase {PHASE}...")
print(f"Config: {train_config}")

In [ ]:
# Run training
results = train_phase(
    training_client=training_client,
    data_iterator=data_iterator,
    renderer=renderer,
    config=train_config,
    checkpoint_callback=on_checkpoint,
)

print(f"\nTraining complete!")
print(f"Steps: {results['steps']}")
print(f"Avg Loss: {results['avg_loss']:.4f}")

## 6. Save Final Model

In [ ]:
# Save final checkpoint
final_path = training_client.save_state(name="final").result().path
print(f"Final checkpoint: {final_path}")

# Copy to Drive
save_to_drive(final_path, f"swe-finetune/checkpoints/{config.name}/final")

# Save for inference
sampling_client = training_client.save_weights_and_get_sampling_client(
    name=f"{config.name}-final"
)
print(f"Model saved for inference: {config.name}-final")

## 7. Test Inference (Optional)

In [ ]:
from tinker.types import SamplingParams, ModelInput

# Test prompt
test_prompt = "Write a Python function to check if a number is prime."

# Build prompt
messages = [{"role": "user", "content": test_prompt}]
prompt_chunk = renderer.build_generation_prompt(messages)

# Sample
result = sampling_client.sample(
    prompt=ModelInput([prompt_chunk]),
    sampling_params=SamplingParams(
        max_tokens=500,
        temperature=0.7,
        stop=renderer.get_stop_sequences(),
    ),
    num_samples=1,
)

output = tokenizer.decode(result.sequences[0].tokens)
print("Generated:")
print(output)

## Next Steps

After completing all phases:
1. Run SWE-bench evaluation with the fine-tuned model
2. Run TerminalBench evaluation
3. Iterate based on results

**Recommended order**: Phase 1 → 2 → 3 → 4 → 5

**Most important**: Phase 4 (SWE-bench trajectories)